In [2]:
import time
import tracemalloc
import heapq
from collections import deque, defaultdict
import matplotlib.pyplot as plt

# Using tracemalloc as a built-in replacement for memory_profiler
def get_memory_stats(func, *args, **kwargs):
    tracemalloc.start()
    start_time = time.perf_counter()
    
    result = func(*args, **kwargs)
    
    end_time = time.perf_counter()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    # Return result, duration (s), and peak memory (MB)
    return result, end_time - start_time, peak / (1024 * 1024)

print("Environment ready. Built-in memory profiling initialized.")

Environment ready. Built-in memory profiling initialized.


In [3]:
class Graph:
    def __init__(self, nodes):
        self.N = nodes
        self.adj_list = defaultdict(list)
        self.matrix = [[0] * nodes for _ in range(nodes)]

    def add_edge(self, u, v, w=1, directed=False):
        self.adj_list[u].append((v, w))
        self.matrix[u][v] = w
        if not directed:
            self.adj_list[v].append((u, w))
            self.matrix[v][u] = w

# Task 2: BFS and DFS Traversals
def bfs(graph, start):
    visited = [False] * graph.N
    queue = deque([start])
    visited[start] = True
    path = []
    while queue:
        u = queue.popleft()
        path.append(u)
        for v, _ in graph.adj_list[u]:
            if not visited[v]:
                visited[v] = True
                queue.append(v)
    return path

def dfs(graph, start, visited=None, path=None):
    if visited is None: visited = [False] * graph.N
    if path is None: path = []
    visited[start] = True
    path.append(start)
    for v, _ in graph.adj_list[start]:
        if not visited[v]:
            dfs(graph, v, visited, path)
    return path

# Example Usage
social_graph = Graph(5)
for u, v in [(0, 1), (0, 2), (1, 3), (2, 4)]: social_graph.add_edge(u, v)
print(f"BFS Path: {bfs(social_graph, 0)}")
print(f"DFS Path: {dfs(social_graph, 0)}")

BFS Path: [0, 1, 2, 3, 4]
DFS Path: [0, 1, 3, 2, 4]


In [4]:
def topological_sort(graph):
    visited = [False] * graph.N
    stack = []
    def visit(u):
        visited[u] = True
        for v, _ in graph.adj_list[u]:
            if not visited[v]: visit(v)
        stack.append(u)
    for i in range(graph.N):
        if not visited[i]: visit(i)
    return stack[::-1]

task_graph = Graph(4)
task_graph.add_edge(0, 1, directed=True)
task_graph.add_edge(0, 2, directed=True)
task_graph.add_edge(1, 3, directed=True)
print(f"Task Execution Sequence: {topological_sort(task_graph)}")

Task Execution Sequence: [0, 2, 1, 3]


In [5]:
def dijkstra(graph, start):
    distances = [float('inf')] * graph.N
    distances[start] = 0
    pq = [(0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > distances[u]: continue
        for v, weight in graph.adj_list[u]:
            if distances[u] + weight < distances[v]:
                distances[v] = distances[u] + weight
                heapq.heappush(pq, (distances[v], v))
    return distances

# Profile Dijkstra
road_network = Graph(6)
edges = [(0,1,7), (0,2,9), (1,2,10), (1,3,15), (2,3,11), (2,4,2), (4,3,6), (4,5,1)]
for u, v, w in edges: road_network.add_edge(u, v, w)

result, time_taken, mem_peak = get_memory_stats(dijkstra, road_network, 0)
print(f"Shortest Paths: {result}")
print(f"Performance: {time_taken:.6f}s | Peak Memory: {mem_peak:.6f} MB")

Shortest Paths: [0, 7, 9, 17, 11, 12]
Performance: 0.001200s | Peak Memory: 0.000069 MB


In [6]:
def prims_mst(graph):
    total_cost = 0
    visited = [False] * graph.N
    pq = [(0, 0)] 
    while pq:
        w, u = heapq.heappop(pq)
        if visited[u]: continue
        visited[u] = True
        total_cost += w
        for v, weight in graph.adj_list[u]:
            if not visited[v]:
                heapq.heappush(pq, (weight, v))
    return total_cost

result_mst, time_mst, mem_mst = get_memory_stats(prims_mst, road_network)
print(f"Total MST Cost: {result_mst}")
print(f"Performance: {time_mst:.6f}s | Peak Memory: {mem_mst:.6f} MB")

Total MST Cost: 25
Performance: 0.000100s | Peak Memory: 0.000076 MB
